# 03 Leakage Comparison

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv('../dataset/pd_speech_features.csv', header=1)
X = df.drop(['id', 'class'], axis=1)
y = df['class']
groups = df['id']


In [2]:
import xgboost as xgb

def evaluate_cv(X, y, groups, cv_type):
    if cv_type == 'GroupKFold':
        cv = GroupKFold(n_splits=5)
        splits = cv.split(X, y, groups)
    else:
        cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        splits = cv.split(X, y)
        
    metrics = {'acc': [], 'auc': []}
    
    for train_idx, test_idx in splits:
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        vt = VarianceThreshold(threshold=0.01)
        X_train = vt.fit_transform(X_train)
        X_test = vt.transform(X_test)
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        skb = SelectKBest(f_classif, k=50)
        X_train = skb.fit_transform(X_train, y_train)
        X_test = skb.transform(X_test)
        
        smote = SMOTE(random_state=RANDOM_STATE)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        
        model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        
        preds = model.predict(X_test)
        probs = model.predict_proba(X_test)[:, 1]
        
        metrics['acc'].append(accuracy_score(y_test, preds))
        metrics['auc'].append(roc_auc_score(y_test, probs))
        
    return {
        'Accuracy': np.mean(metrics['acc']),
        'AUC': np.mean(metrics['auc'])
    }

res_group = evaluate_cv(X, y, groups, 'GroupKFold')
res_kfold = evaluate_cv(X, y, groups, 'KFold')

inflation_acc = res_kfold['Accuracy'] - res_group['Accuracy']
inflation_auc = res_kfold['AUC'] - res_group['AUC']

print("Leakage Comparison:")
print(f"GroupKFold Accuracy: {res_group['Accuracy']:.4f}, AUC: {res_group['AUC']:.4f}")
print(f"Naive KFold Accuracy: {res_kfold['Accuracy']:.4f}, AUC: {res_kfold['AUC']:.4f}")
print(f"Inflation Accuracy: +{inflation_acc*100:.2f} percentage points")
print(f"Inflation AUC: +{inflation_auc*100:.2f} percentage points")

pd.DataFrame({
    'Metric': ['Accuracy', 'AUC'],
    'Strict GroupKFold': [f"{res_group['Accuracy']:.4f}", f"{res_group['AUC']:.4f}"],
    'Naive KFold (Leaked)': [f"{res_kfold['Accuracy']:.4f}", f"{res_kfold['AUC']:.4f}"],
    'Inflation (pts)': [f"+{inflation_acc*100:.2f}", f"+{inflation_auc*100:.2f}"]
}).to_csv("leakage_comparison.csv", index=False)


Leakage Comparison:
GroupKFold Accuracy: 0.7752, AUC: 0.7802
Naive KFold Accuracy: 0.8400, AUC: 0.8808
Inflation Accuracy: +6.48 percentage points
Inflation AUC: +10.06 percentage points
